<a href="https://colab.research.google.com/github/abhi-mike-g/HyperVergeOT/blob/main/colab_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HV-Doc-Data — Download from Kaggle + Train U-Net

Run **top to bottom** on Google Colab (`Runtime → Change runtime type → T4 GPU`).

**What this notebook does:**
1. Downloads `yashvardhangera/hv-doc-data` from Kaggle via an interactive token prompt
2. Copies the dataset into `/content/` so it's visible in the file browser
3. Picks one of the three training-round CSVs and trains a U-Net
4. Saves a checkpoint, then lets you load **any** checkpoint to generate `pred.csv`

**Before you start:** go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token** — you'll be asked to paste your username and key in the auth cell below.

## 1 — Install dependencies

In [ ]:
!pip install -q kagglehub albumentations tqdm
print("done installing")

## 2 — Kaggle authentication

Run this cell and follow the prompt to paste your Kaggle username and API key.

In [ ]:
import kagglehub

kagglehub.login()

## 3 — Download and copy dataset to `/content/`

`kagglehub` caches to `/root/.cache/...` — not visible in the Colab file browser. This cell downloads the dataset, finds the actual data directory (the Kaggle package nests it under `student/student/`), then copies images and labels into `/content/dataset/` where you can browse them.

In [ ]:
import kagglehub, os, shutil, glob

# Step 1 — download to the kagglehub cache (~/.cache/kagglehub/...)
_cache_path = kagglehub.dataset_download("yashvardhangera/hv-doc-data")
print("Kaggle cache path:", _cache_path)

# Step 2 — find the subdirectory that actually contains images/ and labels/
# (the Kaggle package nests the data under student/student/)
def _find_data_root(base):
    for root, dirs, _ in os.walk(base):
        if "images" in dirs and "labels" in dirs:
            return root
    raise RuntimeError(f"Could not find a folder with both images/ and labels/ under {base}")

_data_root = _find_data_root(_cache_path)
print("Data root found at:", _data_root)

# Step 3 — copy into /content/dataset/ so it's visible in the file browser
CONTENT_DATA = "/content/dataset"

if os.path.exists(CONTENT_DATA):
    print(f"{CONTENT_DATA} already exists, skipping copy")
else:
    print("Copying to /content/dataset/ ...")
    shutil.copytree(_data_root, CONTENT_DATA)
    print("Done.")

# Step 4 — copy seg_common.py to /content/ so it's importable
_sc_src = os.path.join(_cache_path, "seg_common.py")
if os.path.exists(_sc_src) and not os.path.exists("/content/seg_common.py"):
    shutil.copy(_sc_src, "/content/seg_common.py")
    print("Copied seg_common.py → /content/seg_common.py")

# Verify
print("\n/content/dataset/ layout:")
for root, dirs, files in os.walk(CONTENT_DATA):
    depth = root[len(CONTENT_DATA):].count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in sorted(files)[:4]:
        print(f"{indent}  {f}")
    if len(files) > 4:
        print(f"{indent}  ... ({len(files)} files total)")

## 4 — Index all images and CSVs

In [ ]:
ALL_CSVS = sorted(glob.glob(os.path.join(CONTENT_DATA, "**", "*.csv"), recursive=True))
ALL_IMAGES = sorted(
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.jpg"),  recursive=True) +
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.jpeg"), recursive=True) +
    glob.glob(os.path.join(CONTENT_DATA, "**", "*.png"),  recursive=True)
)

print(f"CSVs  : {len(ALL_CSVS)}")
for p in ALL_CSVS:
    print(" ", p)
print(f"Images: {len(ALL_IMAGES)}  (first few below)")
for p in ALL_IMAGES[:4]:
    print(" ", p)

assert ALL_CSVS,   "No CSV files found under /content/dataset/"
assert ALL_IMAGES, "No image files found under /content/dataset/"

# filename -> full /content/ path
image_lookup = {os.path.basename(p): p for p in ALL_IMAGES}

# test images (used later for pred.csv)
TEST_IMAGE_PATHS = sorted(p for p in ALL_IMAGES if os.path.basename(p).startswith("test_"))
print(f"\nTrain images: {sum(1 for p in ALL_IMAGES if os.path.basename(p).startswith('train_'))}")
print(f"Test  images: {len(TEST_IMAGE_PATHS)}")

## 5 — Pick training round and load labels

Set `TRAIN_ROUND` to **1, 2, or 3** to choose which polluted label set to train on.

In [ ]:
import pandas as pd, json as _json

# ── Pick your training round ──────────────────────────────────────────────────
TRAIN_ROUND = 1   # 1, 2, or 3
# ─────────────────────────────────────────────────────────────────────────────

target_csv = f"train_round_{TRAIN_ROUND}.csv"
matches = [p for p in ALL_CSVS if os.path.basename(p) == target_csv]
assert matches, f"Could not find {target_csv} among: {ALL_CSVS}"
LABELS_CSV_PATH = matches[0]

labels_df = pd.read_csv(LABELS_CSV_PATH)
assert "image"   in labels_df.columns, f"Expected 'image' column, got: {list(labels_df.columns)}"
assert "polygon" in labels_df.columns, f"Expected 'polygon' column, got: {list(labels_df.columns)}"
labels_df = labels_df.drop_duplicates(subset="image").reset_index(drop=True)

missing = [n for n in labels_df["image"] if n not in image_lookup]
print(f"Round {TRAIN_ROUND}: {len(labels_df)} images | {len(missing)} missing from disk")
print(f"Labels CSV: {LABELS_CSV_PATH}")

## 6 — Write `seg_common.py`

In [ ]:
%%writefile /content/seg_common.py
"""Shared U-Net code for the document segmentation task."""
import csv, glob, json, os, random
import cv2, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset

DATA_ROOT = "/content/dataset"
PRED_ROOT = "/content/results"
CKPT_ROOT = "/content/checkpoints"

IMG_SIZE = 256
IOU_THR  = 0.75
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.d1 = DoubleConv(3, base);        self.d2 = DoubleConv(base, base*2)
        self.d3 = DoubleConv(base*2, base*4); self.d4 = DoubleConv(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2); self.u3 = DoubleConv(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2); self.u2 = DoubleConv(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base,   2, stride=2); self.u1 = DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.d1(x); e2 = self.d2(self.pool(e1))
        e3 = self.d3(self.pool(e2)); bn = self.d4(self.pool(e3))
        d3 = self.u3(torch.cat([self.up3(bn), e3], 1))
        d2 = self.u2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.u1(torch.cat([self.up1(d2), e1], 1))
        return self.out(d1)


def polys_to_mask(polygons, H, W):
    mask = np.zeros((H, W), np.uint8)
    for poly in polygons:
        if poly and len(poly) >= 3:
            pts = (np.array(poly, np.float32) * np.array([W, H])).astype(np.int32)
            cv2.fillPoly(mask, [pts], 255)
    return mask


class CsvSegDataset(Dataset):
    """Records: list of {image_name, image_path, polygons}."""
    def __init__(self, records, transform):
        self.records = records; self.transform = transform
    def __len__(self): return len(self.records)
    def _raw(self, i):
        r = self.records[i]
        img  = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        mask = polys_to_mask(r["polygons"], H, W)
        return img, mask
    def __getitem__(self, i):
        img, mask = self._raw(i)
        t = self.transform(image=img, mask=mask)
        return t["image"], (t["mask"] > 127).float().unsqueeze(0)


def dice_coeff(pred, gt, eps=1e-6):
    pred = np.asarray(pred).astype(bool); gt = np.asarray(gt).astype(bool)
    return float((2 * np.logical_and(pred, gt).sum() + eps) / (pred.sum() + gt.sum() + eps))


def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    num  = 2 * (prob * target).sum(dim=(1,2,3)) + eps
    den  = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + eps
    return (1 - num / den).mean()


def preprocess(img_rgb):
    r = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    n = (r.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(n.transpose(2, 0, 1)).unsqueeze(0).float()


def mask_to_polygons(binary, size=None, min_area=80, epsilon_frac=0.005):
    if size is None: size = IMG_SIZE
    contours, _ = cv2.findContours(binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polygons = []
    for c in contours:
        if cv2.contourArea(c) < min_area: continue
        approx = cv2.approxPolyDP(c, epsilon_frac * cv2.arcLength(c, True), True)
        if len(approx) < 3: continue
        pts = approx.reshape(-1, 2).astype(np.float32) / size
        polygons.append([[round(float(x),5), round(float(y),5)] for x,y in pts])
    return polygons

## 7 — Imports and config

In [ ]:
import sys, random, csv, json
import numpy as np, cv2, torch, torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, "/content")
import seg_common as sc

os.makedirs(sc.CKPT_ROOT, exist_ok=True)
os.makedirs(sc.PRED_ROOT, exist_ok=True)

torch.manual_seed(0); np.random.seed(0); random.seed(0)
print("device:", sc.DEVICE, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 8 — Training hyper-parameters

In [ ]:
MODEL_NAME  = f"unet_round{TRAIN_ROUND}"
BATCH       = 16
EPOCHS      = 1
LR          = 1e-3
VAL_FRAC    = 0.10
NUM_WORKERS = 2

print(f"Round {TRAIN_ROUND}  |  model: {MODEL_NAME}  |  img_size {sc.IMG_SIZE}  |  batch {BATCH}  |  epochs {EPOCHS}")

## 9 — Transforms

In [ ]:
train_tf = A.Compose([
    A.Resize(sc.IMG_SIZE, sc.IMG_SIZE),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(sc.IMG_SIZE, sc.IMG_SIZE),
    A.Normalize(mean=sc.MEAN, std=sc.STD),
    ToTensorV2(),
])

## 10 — Build train / val split

In [ ]:
records, skipped = [], 0
for row in labels_df.itertuples(index=False):
    name = row.image
    if name not in image_lookup:
        skipped += 1; continue
    records.append({
        "image_name": name,
        "image_path": image_lookup[name],
        "polygons":   _json.loads(row.polygon) if isinstance(row.polygon, str) else row.polygon,
    })
if skipped:
    print(f"Skipped {skipped} label rows with no matching image on disk")

random.Random(0).shuffle(records)
n_val         = int(len(records) * VAL_FRAC)
val_records   = records[:n_val]
train_records = records[n_val:]

train_ds = sc.CsvSegDataset(train_records, train_tf)
val_ds   = sc.CsvSegDataset(val_records,   val_tf)
train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_ld   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"train: {len(train_ds)} images  |  val: {len(val_ds)} images")

## 11 — Model

In [ ]:
model = sc.UNet(base=32).to(sc.DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
bce   = nn.BCEWithLogitsLoss()

def loss_fn(logit, target):
    return bce(logit, target) + sc.dice_loss(logit, target)

print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params  |  device: {sc.DEVICE}")

## 12 — Train

In [ ]:
@torch.no_grad()
def eval_dice(loader):
    model.eval()
    scores = []
    for x, y in loader:
        prob = torch.sigmoid(model(x.to(sc.DEVICE)))
        for p, g in zip(prob.cpu().numpy(), y.numpy()):
            scores.append(sc.dice_coeff(p > 0.5, g))
    return float(np.mean(scores))


hist = {"train_loss": [], "val_dice": []}

for ep in range(1, EPOCHS + 1):
    model.train()
    running, n = 0.0, 0
    pbar = tqdm(train_ld, desc=f"epoch {ep}/{EPOCHS}", leave=False)
    for x, y in pbar:
        x, y = x.to(sc.DEVICE), y.to(sc.DEVICE)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        running += loss.item(); n += 1
        pbar.set_postfix(loss=running / n)
    sched.step()
    val_dice = eval_dice(val_ld)
    hist["train_loss"].append(running / n)
    hist["val_dice"].append(val_dice)
    print(f"epoch {ep:2d}  loss {running/n:.4f}  val_dice {val_dice:.4f}")

## 13 — Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist["train_loss"], marker="o"); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(hist["val_dice"],   marker="o"); ax[1].set_title("val Dice");   ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 14 — Save checkpoint

In [ ]:
CKPT_PATH = os.path.join(sc.CKPT_ROOT, f"{MODEL_NAME}.pt")
torch.save(model.state_dict(), CKPT_PATH)
print("Checkpoint saved →", CKPT_PATH)

---
## 15 — Generate `pred.csv`

You can run this section **independently** after a session reset — just re-run Sections 1–7 first.

Set `CKPT_PATH` to whichever checkpoint file you want to use.

In [ ]:
# ── Set the path to the checkpoint you want to use ───────────────────────────
CKPT_PATH = "/content/checkpoints/unet_round1.pt"
# ─────────────────────────────────────────────────────────────────────────────

model = sc.UNet(base=32).to(sc.DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=sc.DEVICE))
model.eval()
print(f"Loaded: {CKPT_PATH}  |  device: {sc.DEVICE}")

In [ ]:
THRESH = 0.5

if not TEST_IMAGE_PATHS:
    print("No test images found in /content/dataset/. Check the dataset structure.")
else:
    print(f"Running inference on {len(TEST_IMAGE_PATHS)} test images...")
    rows = []
    with torch.no_grad():
        for img_path in tqdm(TEST_IMAGE_PATHS, desc="predicting"):
            image    = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
            prob     = torch.sigmoid(model(sc.preprocess(image).to(sc.DEVICE)))[0, 0].cpu().numpy()
            polygons = sc.mask_to_polygons((prob > THRESH).astype(np.uint8))
            rows.append((os.path.basename(img_path), json.dumps(polygons)))

    PRED_CSV = "/content/results/pred.csv"
    with open(PRED_CSV, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image", "polygon"])
        w.writerows(rows)
    print(f"Wrote {len(rows)} rows → {PRED_CSV}")

## 16 — Download `pred.csv`

In [ ]:
import requests

# ── CONFIG: edit these ────────────────────────────────────────────────────────

BASE_URL        = "http://3.6.116.106:8991"
HIRING_CODE     = "College_code_here"

PREDICTIONS_CSV = "/content/results/pred.csv"        # path to YOUR predictions
NAME            = "Test_HV"
EMAIL           = "yashvardhan@university.edu"     # your unique ID -- use the same one every time

# ──────────────────────────────────────────────────────────────────────────────


def submit_predictions(pred_path: str, name: str, email: str, hiring_code: str) -> dict:
    url = f"{BASE_URL}/submit"
    with open(pred_path, "rb") as f:
        files = {"predictions": (pred_path, f, "text/csv")}
        data = {"name": name, "email": email, "hiring_code": hiring_code}
        resp = requests.post(url, data=data, files=files, timeout=300)

    if not resp.ok:
        print(f"Request failed: {resp.status_code}")
        try:
            print(resp.json().get("detail", resp.text))
        except ValueError:
            print(resp.text)
        return {}

    return resp.json()


def print_results(result: dict) -> None:
    if not result:
        return

    s = result["this_submission"]
    b = result["your_best"]

    print()
    print(f'Images evaluated   : {s["n_images"]}')
    print(f'Instance F1       : {s["f1"]:.4f}  (IoU thr = {s["iou_thr"]:.2f})')
    print(f'Instance Precision : {s["precision"]:.4f}  (TP={s["tp"]}  FP={s["fp"]})')
    print(f'Instance Recall    : {s["recall"]:.4f}  (FN={s["fn"]})')
    print(f'Pixel Dice (mean)  : {s["dice_mean"]:.4f}')
    print()
    print(f'Attempts so far    : {result["attempt"]}')
    print(f'Attempts remaining : {result["attempts_remaining"]}')


print_results(submit_predictions(
    PREDICTIONS_CSV,
    NAME,
    EMAIL,
    HIRING_CODE,
))

In [ ]:
from google.colab import files
files.download(PRED_CSV)